In [23]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
%cd ..

c:\Users\arik_\Documents\Dokumente\Job_Clausthal\TNTM\TNTM_Revision_TNNLS


In [25]:
import torch
import pickle
from Benchmark.TopMost2OctisAdapter import TopMost2OctisAdapter

from topmost.models import ProdLDA

In [26]:


#with open("C:\\Users\\arik_\\\Documents\\Dokumente\\Job_Clausthal\\TNTM\\TNTM_Revision_TNNLS\\TNTM\\Data\\DataOctis2\\octis_dataset_20ng.pickle", "rb") as file:
#    bow_data = pickle.load(file)

with open("C:\\Users\\arik_\\Documents\\Dokumente\\Job_Clausthal\\TNTM\\TNTM_Revision_TNNLS\\TNTM\Data\DataOctis2\\octis_dataset_20ng.pickle", "rb") as file:
    octis_dataset = pickle.load(file)



In [39]:
from Code.Evaluate.Metrics import score_all, get_tw_embeddings

corpus = octis_dataset.get_corpus()
tw_emb = get_tw_embeddings(octis_dataset)
with open("C:\\Users\\arik_\\Documents\\Dokumente\\Job_Clausthal\\TNTM\\TNTM_Revision_TNNLS\\TNTM\Data\DataOctis\\cleaned_embedding_df_20ng_BERT.pickle", 'rb') as f:
    embedding_df = pickle.load(f)
embedding_df.sort_values(by = "word", inplace = True)
embedding_ten_lis = []

for i in range(len(embedding_df)):
    embedding_ten_lis.append(embedding_df["embedding"].iloc[i])
embedding_df.sort_values(by = "word", inplace = True)
embedding_ten_lis = []

embedded_words = embedding_df.index.tolist()

for i in range(len(embedding_df)):
    embedding_ten_lis.append(embedding_df["embedding"].iloc[i])
embedding_ten = torch.stack(embedding_ten_lis)  

100%|██████████| 48018/48018 [1:26:07<00:00,  9.29it/s]     


In [28]:
corpus = octis_dataset.get_corpus()
vocab = octis_dataset.get_vocabulary()

In [29]:
len(corpus), len(vocab)

(18846, 3350)

In [30]:
corpus_red = corpus[:100]
vocab_red = [word for doc in corpus_red for word in doc]
vocab_red = list(set(vocab_red))

In [31]:
octis_dataset._Dataset__corpus = corpus_red
octis_dataset._Dataset__vocab = vocab_red

In [32]:
def data2params_prodlda(corpus, vocab):
    return {
        "vocab_size": len(vocab),
    }



In [33]:
model = TopMost2OctisAdapter(
    model_topmost = ProdLDA,
    model_kwargs= {
        "num_topics": 10
    },
    data2_additional_kwargs = data2params_prodlda,
    batch_size = 512
)

In [34]:
len(vocab_red)

2449

In [35]:
res = model.fit(octis_dataset)

loading word embeddings: 100%|██████████| 2449/2449 [00:00<00:00, 7663.95it/s]


2449


100%|██████████| 200/200 [00:03<00:00, 57.17it/s]


In [41]:
evaluation_result = score_all(
    dataset = octis_dataset,
    tw_emb=tw_emb,
    n_words=10,
    result = {'topics': res[0], 
              "topic-word-matrix": res[1]},
)

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\arik_\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arik_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
100%|██████████| 2/2 [00:41<00:00, 20.62s/it]


In [42]:
evaluation_result

{'NPMI': 0.2778632685092054, 'WE_CO_PW': 0.21}

In [45]:
from Benchmark.Benchmark import Benchmark

In [46]:
bench = Benchmark(
    octis_dataset = octis_dataset,
    embedding_df = embedding_df,
    models = [model],
    model_specific_data2params_fun_list = [[data2params_prodlda]],
    n_topics = [10, 15],
    batch_size = 256
)

In [47]:
bench.run()

 52%|█████▏    | 24906/48018 [05:30<10:35, 36.36it/s]